In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [4]:

 
# Show all columns when printing, so nothing is hidden as "..."
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [7]:
# STEP 2: read the file
# What: the CSV becomes a table in Python (10,889 rows).
#       The r before the path tells Python to read the
#       backslashes (\) exactly as they are.
# Why:  the original file is only read and stays unchanged,
#       so we can always go back to the full data.

df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")
 
print("STEP 2: original data read from CSV file")
print("Rows and columns:", df.shape)
print(df.head())

STEP 2: original data read from CSV file
Rows and columns: (10889, 7)
   year_film  year_ceremony  ceremony category                 name              film  winner
0       1927           1928         1    ACTOR  Richard Barthelmess         The Noose   False
1       1927           1928         1    ACTOR        Emil Jannings  The Last Command    True
2       1927           1928         1  ACTRESS       Louise Dresser   A Ship Comes In   False
3       1927           1928         1  ACTRESS         Janet Gaynor        7th Heaven    True
4       1927           1928         1  ACTRESS       Gloria Swanson    Sadie Thompson   False


In [9]:

# STEP 3: keep only the winners with a film
# What: only rows where "winner" is True and "film" is not
#       empty are kept.
# Why:  our research looks at the effect of WINNING an Oscar
#       on box office. Every directing award belongs to a film.
# ------------------------------------------------------------
winners = df[(df["winner"] == True) & (df["film"].notna())]
 
print("STEP 3: only winners with a film are kept")
print("Rows before:", len(df))
print("Rows after: ", len(winners))
 

STEP 3: only winners with a film are kept
Rows before: 10889
Rows after:  2172


In [10]:
# STEP 4: keep only the directing awards
# What: only rows whose category starts with DIRECTING are
#       kept.
# Why:  AWARD_Director only holds directing Oscars.
#       We use startswith on purpose: searching for "DIRECT"
#       would also keep Art Direction, Assistant Director and
#       Dance Direction, which are NOT directing Oscars.
# ------------------------------------------------------------
award_director = winners[winners["category"].str.startswith("DIRECTING")].copy()
 
print("STEP 4: only directing awards are kept")
print("Rows before:", len(winners))
print("Rows after: ", len(award_director))
print("Categories that are kept:", list(award_director["category"].unique()))
 

STEP 4: only directing awards are kept
Rows before: 2172
Rows after:  97
Categories that are kept: ['DIRECTING (Comedy Picture)', 'DIRECTING (Dramatic Picture)', 'DIRECTING']


In [13]:

# STEP 5: match the column names to the ERD
# What: "name" becomes "name_nominee", "film" becomes
#       "film_title".
# Why:  this way the columns match the table in PostgreSQL.
# ------------------------------------------------------------
print("STEP 5: rename columns to match the ERD")
print("Columns before:", list(award_director.columns))
 
award_director = award_director.rename(columns={"name": "name_nominee", "film": "film_title"})
 
print("Columns after: ", list(award_director.columns))
 

STEP 5: rename columns to match the ERD
Columns before: ['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']
Columns after:  ['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']


In [14]:
# STEP 6: remove the ";" from the film titles
# What: rstrip(";") removes a ";" at the end of the title,
#       strip() removes spaces at the beginning and end.
# Why:  some old titles in the source data end with a ";",
#       which is a leftover and not part of the title.
#       We clean it here too, so all award tables are cleaned
#       in the same way.
# ------------------------------------------------------------
print("STEP 6: remove ';' from titles")
print("Titles with ';' before:", award_director["film_title"].str.contains(";").sum())
 
award_director["film_title"] = award_director["film_title"].str.rstrip(";").str.strip()
 
print("Titles with ';' after: ", award_director["film_title"].str.contains(";").sum())

STEP 6: remove ';' from titles
Titles with ';' before: 0
Titles with ';' after:  0


In [15]:

# STEP 7: sort the table
# What: the rows are sorted by year_ceremony, then by category
#       and then by film_title.
# Why:  a clear, fixed order makes the table easy to read and
#       check, and gives the same award_id every time the
#       script runs.
# ------------------------------------------------------------
award_director = award_director.sort_values(["year_ceremony", "category", "film_title"])
 
print("\n=== STEP 7: sorted table ===")
print(award_director[["year_ceremony", "category", "name_nominee", "film_title"]].head())


=== STEP 7: sorted table ===
     year_ceremony                      category     name_nominee                      film_title
11            1928    DIRECTING (Comedy Picture)  Lewis Milestone             Two Arabian Knights
13            1928  DIRECTING (Dramatic Picture)    Frank Borzage                      7th Heaven
59            1929                     DIRECTING      Frank Lloyd                 The Divine Lady
98            1930                     DIRECTING  Lewis Milestone  All Quiet on the Western Front
139           1931                     DIRECTING    Norman Taurog                          Skippy


In [16]:
# STEP 8: give every award a unique award_id
# What: reset_index renumbers the rows 0, 1, 2... Then every
#       award gets a number from 1 to 97.
# Why:  award_id is the primary key of AWARD_Director.
#       A primary key must be unique and cannot be empty.
# ------------------------------------------------------------
award_director = award_director.reset_index(drop=True)
award_director["award_id"] = range(1, len(award_director) + 1)
 
print("\n=== STEP 8: award_id added ===")
print("First award_id:", award_director["award_id"].min())
print("Last award_id: ", award_director["award_id"].max())
print("All unique:    ", award_director["award_id"].is_unique)


=== STEP 8: award_id added ===
First award_id: 1
Last award_id:  97
All unique:     True


In [17]:
# STEP 8: give every award a unique award_id
# What: reset_index renumbers the rows 0, 1, 2... Then every
#       award gets a number from 1 to 97.
# Why:  award_id is the primary key of AWARD_Director.
#       A primary key must be unique and cannot be empty.
# ------------------------------------------------------------
award_director = award_director.reset_index(drop=True)
award_director["award_id"] = range(1, len(award_director) + 1)
 
print(" STEP 8: award_id added ")
print("First award_id:", award_director["award_id"].min())
print("Last award_id: ", award_director["award_id"].max())
print("All unique:    ", award_director["award_id"].is_unique)

 STEP 8: award_id added 
First award_id: 1
Last award_id:  97
All unique:     True


In [18]:
 # STEP 9: add empty director_id and movie_id columns
# What: the table gets a director_id and a movie_id column
#       without values.
# Why:  both are in our ERD as foreign keys (to DIRECTOR and
#       to MOVIE), so the columns must exist. They stay empty
#       for now and are filled in later. Int64 keeps them
#       whole numbers.
# ------------------------------------------------------------
award_director["director_id"] = pd.Series(dtype="Int64")
award_director["movie_id"] = pd.Series(dtype="Int64")
 
print(" STEP 9: empty director_id and movie_id added ")
print("Empty director_id values:", award_director["director_id"].isna().sum(), "of", len(award_director))
print("Empty movie_id values:   ", award_director["movie_id"].isna().sum(), "of", len(award_director))
 

 STEP 9: empty director_id and movie_id added 
Empty director_id values: 97 of 97
Empty movie_id values:    97 of 97


In [19]:
# STEP 10: keep only the ERD columns, in the ERD order
# What: only the columns from the ERD are kept.
# Why:  so the CSV fits the AWARD_Director table in
#       PostgreSQL. "winner" is dropped (every row is a
#       winner) and "year_film" is dropped (it is always
#       year_ceremony - 1).
# ------------------------------------------------------------
award_director = award_director[["award_id", "director_id", "movie_id", "year_ceremony",
                                 "ceremony", "category", "name_nominee", "film_title"]]
 
print(" STEP 10: final columns ")
print(list(award_director.columns))
 

 STEP 10: final columns 
['award_id', 'director_id', 'movie_id', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title']


In [23]:
 # STEP 11: save as a CSV file
# What: the table is saved as award_director.csv in the same
#       folder as the original file.
# Why:  this file is imported into AWARD_Director in pgAdmin.
#       A new file name, so the original is not overwritten.
#       index=False prevents an extra column with row numbers.
# ------------------------------------------------------------

import os

folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_director.clean.csv")
 
try:
    award_director.to_csv(new_file, index=False)
    print(" STEP 11: saved")
    print("File saved as:", new_file)
    print("File exists:", os.path.exists(new_file))
except PermissionError:
    print("ERROR: the file is open in Excel. Close it and run this cell again.")
except OSError as error:
    print("ERROR: something is wrong with the folder or name:", error)
 
 

 STEP 11: saved
File saved as: C:\Users\caple\OneDrive - HvA\Master\Data management\Data\award_director.clean.csv
File exists: True
